# 09 — Physical characterization of the anomaly populations

Characterize the four represented 99x subtypes in the processed pool: binary
microlensing (991), intermediate-luminosity optical transients (992), calcium-rich
transients (993) and pair-instability supernovae (994). These identities follow
the PLAsTiCC model definitions; they are not inferred from this diagnostic.


## Methodological boundary

The thirteen auxiliary physical descriptors are excluded from the 304 model
inputs, AL selection and performance metrics. They are used only after selection
for interpretation. Distance and rest-frame proxies depend on photometric
redshift and are subject to its uncertainties and simplified corrections.


## 0. Configuration


In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'config.py').exists() and (p/'src').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import config
from src.models import classification
from src.models import anomalies as an
from src import physical_analysis as af
from src.interpretability import shap_utils as su
su.thesis_plot_style()
RS = config.RANDOM_STATE
FIG = config.FIGURE_DIR


## 1. Load the model, pool labels and interpretation features


In [ ]:

tr = classification.features_e2gp('train')
X_tr, COLS = classification.e2gp_matrix(tr); y_tr = tr['target'].to_numpy()
RETRAIN_MODEL = False
if RETRAIN_MODEL:
    clf = classification.e2gp_al(y=y_tr, random_state=RS).fit(X_tr, y_tr)
else:
    clf, saved_columns, model_contract = classification.load_e2gp_artifact()
    if list(COLS) != list(saved_columns):
        raise ValueError('Current features do not match the saved model contract')
    print('Loaded the verified canonical model; classifier fitting was not repeated.')
meta_glob = pd.read_csv(config.TEST_METADATA_CSV,
                        usecols=['object_id', 'true_target', 'hostgal_photoz', 'distmod'])


anom = (pd.read_csv(config.DIR_PROCESSED / 'features_extra_test_b99.csv')
          .merge(meta_glob[['object_id', 'true_target', 'hostgal_photoz']], on='object_id'))
anom = anom[anom.true_target >= 990].copy()


ex_tr = (pd.read_csv(config.DIR_PROCESSED / 'features_extra.csv')
           .merge(pd.read_csv(config.METADATA_CLEAN_CSV)[['object_id', 'target']], on='object_id'))

print('Pool anomalies with physical descriptors:', len(anom), ' -> ',
      ', '.join(f'{sm}:{int((anom.true_target == sm).sum())}' for sm in [991, 992, 993, 994]))


## 2. Median subtype properties


In [ ]:
af.property_table(anom)


## 3. Luminosity–duration diagram


In [ ]:
af.phase_diagram(ex_tr, anom, save=FIG / '09_diagrama_fase.png')
plt.show()


## 4. Galactic and extragalactic redshift distributions


In [ ]:
af.subtype_redshifts(anom, save=FIG / '09_redshift.png')
plt.show()


## 5. Descriptor–subtype complementarity


In [ ]:

sc, y_bin, tt = an.batch_descriptors(99, clf, COLS, meta_glob)
rank_df = pd.DataFrame({
    'novedad_rank': an.rank01(sc['novedad_1-conf']),
    'iso_rank':     an.rank01(sc['IsolationForest']),
    'gp_rank':      an.rank01(sc['residuos_GP']),
    'mag_rank':     an.rank01(sc['cambios_mag']),
})
M = af.descriptor_heatmap(rank_df, tt, save=FIG / '09_descriptor_heatmap.png')
plt.show()
pd.DataFrame(M, index=['novedad', 'iso', 'gp', 'mag'],
             columns=[af.NOMBRE99[c] for c in [991, 992, 993, 994]]).round(2)


## 6. Illustrative light curves


In [ ]:

rutas_b4 = config.BatchConfig(4)
lc = pd.read_csv(rutas_b4.test_clean_csv)
meta_b4 = pd.read_csv(rutas_b4.test_metadata_clean_csv)[['object_id', 'true_target']]
det = lc[lc.detected == 1]
ndet = det.groupby('object_id').size()
ampl = det.groupby('object_id').flux.agg(lambda x: x.max() - x.min())
bien = ndet[ndet >= 20].index
ejemplos = {}
for sm in [991, 992, 993, 994]:
    object_ids = set(meta_b4.loc[meta_b4.true_target == sm, 'object_id']) & set(bien)
    if object_ids:
        ejemplos[sm] = int(ampl[ampl.index.isin(object_ids)].idxmax())
af.light_curves(lc[lc.object_id.isin(ejemplos.values())], ejemplos,
              save=FIG / '09_curvas_luz.png')
plt.show()
print('Example objects (batch 4):', ejemplos)


## Interpretation

Relate population-level physical distributions to descriptor complementarity
and confidence-based removal. Example light curves are illustrative rather
than representative samples. Brightness or duration outliers alone do not
establish an object's identity, and photometric-redshift errors can create
extreme inferred absolute magnitudes.
